# Feature Clustering Experiments

This notebook explores three different clustering algorithms to reconstruct "Features" from raw developer events. We compare Connected Components (Baseline), DBSCAN, and Graph Community Detection (Louvain) to see which handles overlapping features and dense subgraphs best.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import networkx as nx
from networkx.algorithms.community import louvain_communities
from sklearn.cluster import DBSCAN
import numpy as np
import itertools
from collections import defaultdict

from synthetic_data import generate_synthetic_events
from feature_reconstruction_simulator import calculate_similarity, evaluate_clusters

np.random.seed(42)
def evaluate_clusters_for_notebook(clusters, total_events):
    correct = 0
    for i, cluster in enumerate(clusters):
        truth_counts = {}
        for e in cluster:
            truth_counts[e['ground_truth']] = truth_counts.get(e['ground_truth'], 0) + 1
        dominant_truth = max(truth_counts.items(), key=lambda x: x[1])[0]
        correct += truth_counts[dominant_truth]
        if len(cluster) > 5:
            print(f"Predicted Feature {i+1} (Size: {len(cluster):>3}) -> Dominant Truth: '{dominant_truth:<20}' (Purity: {truth_counts[dominant_truth]}/{len(cluster)})")
    accuracy = correct / total_events
    print(f"\nTotal Overall Reconstruction Purity: {accuracy*100:.2f}%")

events = generate_synthetic_events(1000)
print("Generated 1000 synthetic events.")

## 1. Baseline: Connected Components

Connects any two events that pass the similarity threshold. Very vulnerable to "bridges" (e.g. `auth.ts` merging "Authentication" and "SSO Migration").

In [ ]:
def build_baseline_clusters(events, threshold=0.39):
    parent = {e['event_id']: e['event_id'] for e in events}
    def find(i):
        if parent[i] == i: return i
        parent[i] = find(parent[i])
        return parent[i]
    def union(i, j):
        root_i = find(i)
        root_j = find(j)
        if root_i != root_j:
            parent[root_i] = root_j
            
    for e1, e2 in itertools.combinations(events, 2):
        if calculate_similarity(e1, e2) >= threshold:
            union(e1['event_id'], e2['event_id'])
            
    clusters = defaultdict(list)
    for e in events:
        clusters[find(e['event_id'])].append(e)
    return sorted(list(clusters.values()), key=len, reverse=True)

clusters_baseline = build_baseline_clusters(events, threshold=0.39)
print("--- Baseline Results ---")
evaluate_clusters_for_notebook(clusters_baseline, len(events))

## 2. DBSCAN

Density-based spatial clustering. Requires a distance matrix (1.0 - similarity).

In [ ]:
def build_dbscan_clusters(events, eps=0.65, min_samples=3):
    n = len(events)
    if n == 0: return []
    dist_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            sim = calculate_similarity(events[i], events[j])
            dist = max(0.0, 1.0 - sim)
            dist_matrix[i, j] = dist
            dist_matrix[j, i] = dist
            
    db = DBSCAN(eps=eps, min_samples=min_samples, metric='precomputed')
    labels = db.fit_predict(dist_matrix)
    
    clusters = defaultdict(list)
    for i, label in enumerate(labels):
        if label != -1:
            clusters[label].append(events[i])
    return sorted(list(clusters.values()), key=len, reverse=True)

clusters_dbscan = build_dbscan_clusters(events, eps=0.65)
print("--- DBSCAN Results ---")
evaluate_clusters_for_notebook(clusters_dbscan, len(events))

## 3. Louvain Community Detection

Graph-based community detection using modularity maximization. Expects dense subgroups of strongly connected events.

In [ ]:
def build_louvain_clusters(events, threshold=0.2):
    G = nx.Graph()
    for e in events:
        G.add_node(e['event_id'], data=e)
        
    for e1, e2 in itertools.combinations(events, 2):
        sim = calculate_similarity(e1, e2)
        if sim > threshold:
            G.add_edge(e1['event_id'], e2['event_id'], weight=sim)
            
    communities = louvain_communities(G, weight='weight')
    
    event_dict = {e['event_id']: e for e in events}
    clusters = []
    for comm in communities:
        cluster = [event_dict[node] for node in comm]
        clusters.append(cluster)
    return sorted(clusters, key=len, reverse=True)

clusters_louvain = build_louvain_clusters(events, threshold=0.2)
print("--- Louvain Community Detection Results ---")
evaluate_clusters_for_notebook(clusters_louvain, len(events))